# SparkSQL and DataFrames

The majority of the data that a data scientist deals with is either structured or semistructured. The PySparkSQL module is a higher-level abstraction over PySpark Core for processing structured and semistructured datasets. By using PySparkSQL, we can use SQL and HiveQL code too, which makes this module popular among database programmers and Apache Hive users.

The APIs provided by PySparkSQL are optimized. PySparkSQL can read data from many file types such as CSV files, JSON files, and files from other databases. In SparkSQL we work with DataFrames instead of RDDs which you may have come across before. The DataFrame abstraction is similar to a table in a relational database management system. The DataFrame consists of named columns and is a collection of Row objects. Row objects are defined in PySparkSQL. It should be noted that in the background the DataFrame is implemented based on RDDs so everything we have learned about RDDs also applies.

Todays lab assumes you already familiar with writting SQL queries. If not it is worth spending time to review an online tutorial on SQL. For example: 

https://www.tutorialspoint.com/sql/index.htm

Note: It will take some time to learn SQL if you have never used it before.

## Setup

In this notebook, we will need a few plotting capabilities. Run the following cell and restart the kernel.

In [ ]:
!pip install sparksql-magic plotly pandas "nbformat>=4.2.0"

### Installing Java

In [ ]:
#Checking the installed Java version
!java -version

In [ ]:
!pip install pyspark 

In [ ]:
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless


In [ ]:
!java -version

In [ ]:

# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .master("local[*]")\
        .appName("PySpark DataFrames") \
        .getOrCreate()

## Dataframes

A DataFrame is collection of named columns, let's take a look at how to create a dataframe from some Python data.


## Using Dataframes

In [ ]:
%%sh 
wget https://raw.githubusercontent.com/UmaMaquinaDeOndas/DataBricks-Tutorials/master/adult.data

In [ ]:
# Load data into the big data cluster as a dataframe
df = spark.read.csv('adult.data',header=True, inferSchema = True)
df.printSchema()

In [ ]:
#Select
df.select("age", "education", "income").show()

In [ ]:
# Where
df.where((df.age > 30) & (df.income == '>50K')).show()

In [ ]:
# Filter
df.filter((df.age > 30) & (df.income == '>50K')).show()

## Agregation in dataframes

To get a summarized pattern of data, data scientists perform aggregation on a given
dataset. Summarized patterns are easy to understand. Sometimes the summarization is
done based on the key. To perform aggregation based on the key, we first need to group
the data by key.

In PySparkSQL, grouping by key can be performed by using the groupBy()
function. This function returns the pyspark.sql.group.GroupedData object. After this
GroupedData object is created, we can apply many aggregation functions such as avg(),
sum(), count(), min(), max(), and sum() on GroupedData.

The following lines will download the adult.data file to the node this notebook is running on from the internet. We can then import it to a dataframe, I have printed the schema which for you to review the headings and get some idea of the data:

Note how we were able to directly load a file we downloaded into a dataframe by telling spark to infer the schema of the data and that the text file had headings.

Take a look at the below examples, to aggregate data by group, sort data, describe data.

For each example try and guess what it will do based on the code before you run the cell:

In [ ]:
# CODE HERE

#### Practice exercises

Combine the functions demonstarted above to find the occupation with the highest number of incomes that are >50K:

In [ ]:
# CODE HERE

Which 5 profession works the most hours on average?

(You don't need to only display the top 5 professions by gender to answer this question, but you can if you like).

In [ ]:
# CODE HERE

## Execute SQL and Queries on a DataFrame

To do so, first we can use createOrReplaceTempView(), which creates a temporary view. You may have used views before in SQL but a view is like a tempory table. The DataFrame class provides this function. The life of this view is the same as the SparkSession that creates the DataFrame.

Using SQLContext, we can run SQL commands. In the preceding section of the tutorial, we created a DataFrame named censusDataFrame. We will create a view from this dataframe:

In [ ]:
df.createOrReplaceTempView("censusDataTable")

We can then pass SQL queries through to spark as follows, notice how we need to include native-country due to the hyphen. Note The spark.sql() function returns a DataFrame:

In [ ]:
# CODE HERE

In [ ]:
%load_ext sparksql_magic

In [ ]:
%%sparksql
SELECT age, income, native_country FROM censusDataTable LIMIT 5


You might notice the way the results are displayed is different, we can use the display command in databricks to load results into the notebook. This will be useful later:

In [ ]:
# CODE HERE

##  Perform Data Joining on DataFrames

Often we’re required to combine information from two or more DataFrames or tables. To
do this, we perform a join of DataFrames. Basically, table joining is a SQL term, where we
join two or more tables to get denormalized tables. Join operations on two tables are very
common in data science.

In PySparkSQL, we can perform the following types of joins (the keyword for the join type is in brackets):

•	 Inner join (deafult)

•	 Left outer join (left_outer)

•	 Right outer join (right_outer)

•	 Full outer join (outer)

![image.png](https://cdn.softwaretestinghelp.com/wp-content/qa/uploads/2019/05/Capture-1.jpg)

Below we provide a small example demonstaring the syntax for joining:

In [ ]:
valuesA = [('Pirate',1,'purple'),('Monkey',2,'brown'),('Ninja',3,'black'),('Spaghetti',4,'white')]
DFTableA = spark.createDataFrame(valuesA,['name','id','colour'])
 
valuesB = [('Rutabaga',5,100),('Pirate',1,150),('Ninja',3,35),('Darth Vader',7,55)]
DFTableB = spark.createDataFrame(valuesB,['name','id','price'])
 
DFTableA.show()
DFTableB.show()

Also note how in the above code we create a dataframe from a list by just providing the data and the names for the headings. This can be a faster way for creating dataframes than the more invovlved method we started the lab with.

Two examples of join are shown below, the first assumes we have the same names for our columns, the second does not:

In [ ]:
DFjoined = DFTableA.join(DFTableB, ['id'])
DFjoined.show()

In [ ]:
joined2 = DFTableA.join(DFTableB, DFTableA.id == DFTableB.id)
joined2.show()

Note that we had to specify how we would join the data, we would match by ID on each table.

Here we show how to perform another join types:

In [ ]:
joinedLeft = DFTableA.join(DFTableB, ['name'],how='left_outer')
joinedLeft.show()

The above join was a left join, make sure you are clear which table was considered the left table when making the join (was it tableA or tableB).

Peform a full outer join the two tables, joining on the name column and use how='outer'.

In [ ]:
outer = DFTableA.join(DFTableB, ['name'],how='outer')
outer.show()

### Visualisation

PySpark also has built-in plotting capabilities. They are powered by the ```plotly``` framework and perform a few tricks to make plotting work for larger datasets.

We will need a few installs that handle the plotting in the notebook for us.

In [ ]:
# CODE HERE

In [ ]:
# CODE HERE

In [ ]:
# Scatter plot
# CODE HERE

In [ ]:
# Box Plot
# CODE HERE

We can also perform a few customizations. 

In [ ]:
# CODE HERE